# 05 — Final validation

## Objective
Validate both the populated city-month representation and the exact structural schema requested for `daily_climate`. Tests cover required fields, key uniqueness, coordinates, date types, non-negative precipitation, and missing-value reporting. Source resolution remains an explicit validation limitation: no daily rows can be asserted or created.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name.lower()=="notebooks" else Path.cwd()
sys.path.insert(0,str(ROOT/"src"))
from climate_pipeline.config import RAW_DIR, PROJECT_DIR, FINAL_DAILY_COLUMNS
from climate_pipeline import run_pipeline, validate_city_monthly, validate_daily_schema
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",60)


## Validate the monthly product and report checks

The city-month dataset is the highest temporal resolution actually present in the raw files. Validate its city/state metadata, unique city-month keys, coordinate ranges, and precipitation semantics. Save a tabular report for audit.

In [2]:
results=run_pipeline(RAW_DIR,ROOT)
city_monthly=results["city_monthly"]
checks=validate_city_monthly(city_monthly)
display(checks)
assert checks.passed.all()
print("city-month rows:",len(city_monthly),"unique city-month keys:",city_monthly[["location_id","date"]].drop_duplicates().shape[0])
display(results["missingness"].sort_values("missing_pct",ascending=False).head(20))


,check,passed
0,city field populated,True
1,state field populated,True
2,one row per location-month,True
3,valid latitude,True
4,valid longitude,True
5,date is datetime,True
6,no negative precipitation,True
7,temperature ordering,True


city-month rows: 2496 unique city-month keys: 2496


,column,missing_count,missing_pct
33,pm25,2496,100.00
32,aqi,2496,100.00
26,district,2496,100.00
34,ndvi,2496,100.00
31,humidity_pct,2496,100.00
5,ALLSKY_SFC_UV_INDEX,96,3.85
6,ALLSKY_SRF_ALB,96,3.85
4,ALLSKY_SFC_SW_DNI,96,3.85
0,city_key,0,0.00
1,city,0,0.00


## Validate daily schema separately

The daily table must have the specified 18 columns in order. It is intentionally empty because monthly summaries cannot be defensibly expanded into daily observations. An empty but schema-valid delivery is an honest representation of the available daily data (none).

In [3]:
daily=pd.read_csv(ROOT/"data"/"processed"/"daily_climate.csv",parse_dates=["date"])
daily_checks=validate_daily_schema(daily)
display(daily_checks)
assert daily_checks.passed.all()
assert daily.empty
assert daily.columns.tolist()==FINAL_DAILY_COLUMNS
print("Validated exact daily schema with",len(daily),"rows")


,check,passed
0,required columns and order,True
1,date dtype is datetime compatible,True
2,daily key unique,True
3,coordinates in range,True
4,rainfall non-negative,True
5,humidity within bounds,True
6,temperature ordering,True


Validated exact daily schema with 0 rows


## Coverage and limitations

The reports folder contains file/point inventory, per-city coverage, missingness by field, and the final check results. Available dataset: 8 cities, 89 measurement points, 2,496 city-month observations from 2000–2025. District, relative humidity, AQI, PM2.5, NDVI, and daily observations are unavailable.